In [ ]:
# ============================================================
# 03d_rolling_inference.ipynb
# Rolling window inference using trained Transformer
#
# For each stay, for each hour h (4 to stay_length):
#   - Feed last 24h of vitals to Transformer
#   - Read off 12h-ahead risk score
#   - Assign alert tier: no_alert / high_risk / critical
#
# Output: rolling_predictions.csv
#   columns: stay_id, hour, risk_score, alert_tier,
#            true_label_12h, split
# ============================================================

import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('C:/Users/20220505/Desktop/AI-sepsis')
sys.path.append(str(PROJECT_ROOT / 'src'))

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")

# ── Constants ─────────────────────────────────────────────────
MAX_HOURS      = 200
LOOKBACK_W     = 24      # hours of history fed to model
HERO_HORIZON_H = 12      # predict sepsis within next 12h
MIN_HOUR       = 4       # start predicting from hour 4
NUM_BINS       = 48
time_cuts      = np.linspace(0, 200, NUM_BINS + 1)[1:]

ALERT_THRESHOLDS = {
    'no_alert'  : (0.00, 0.10),
    'high_risk' : (0.10, 0.50),
    'critical'  : (0.50, 1.00),
}
ALERT_COLORS = {
    'no_alert'  : '#27AE60',
    'high_risk' : '#E67E22',
    'critical'  : '#C0392B',
}

def get_alert_tier(score):
    if score >= ALERT_THRESHOLDS['critical'][0]:
        return 'critical'
    elif score >= ALERT_THRESHOLDS['high_risk'][0]:
        return 'high_risk'
    else:
        return 'no_alert'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Load metadata ─────────────────────────────────────────────
print("\nLoading metadata...")

with open(str(OUTPUT_DIR / 'rich_feature_names.txt')) as f:
    rich_feature_names = f.read().splitlines()

with open(str(OUTPUT_DIR / 'feature_names.txt')) as f:
    feature_cols = f.read().splitlines()

stay_ids_order = pd.read_csv(
    str(OUTPUT_DIR / 'stay_ids_order.csv')
).squeeze().tolist()
stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}

print(f"  Rich features : {len(rich_feature_names)}")
print(f"  Static features: {len(feature_cols)}")
print(f"  Stays         : {len(stay_ids_order):,}")

# ── Load X_rich_full ──────────────────────────────────────────
print("\nLoading X_rich_full.npy (1.49 GB)...")
X_rich_full = np.load(str(OUTPUT_DIR / 'X_rich_full.npy'))
print(f"  Shape : {X_rich_full.shape}")

# ── Load static features ──────────────────────────────────────
print("Loading engineered features...")
all_features  = pd.read_csv(str(OUTPUT_DIR / 'engineered_features.csv'))
static_lookup = (
    all_features
    .set_index('stay_id')[feature_cols]
    .fillna(0)
)
print(f"  Shape : {all_features.shape}")

# ── Load hourly labels ────────────────────────────────────────
print("Loading hourly labels...")
hl = pd.read_csv(str(OUTPUT_DIR / 'hourly_labels.csv'))
print(f"  Shape : {hl.shape}")

# ── Load split info ───────────────────────────────────────────
print("Loading splits...")
split_df  = pd.read_csv(str(OUTPUT_DIR / 'subject_splits.csv'))
cohort    = pd.read_csv(
    str(OUTPUT_DIR / 'icu_cohort (1).csv'),
    usecols=['stay_id', 'subject_id']
)
stay_split = (
    cohort
    .merge(split_df, on='subject_id', how='left')
    .set_index('stay_id')['split']
    .to_dict()
)
print(f"  Split counts: "
      f"{pd.Series(stay_split).value_counts().to_dict()}")

print("\nSetup complete ✓")

In [ ]:
# ============================================================
# Cell 2: Rebuild Transformer architecture and load weights
# Must match exactly what was defined in 03b
# ============================================================

class TransformerSurvival(nn.Module):
    def __init__(self,
                 vital_dim    = 25,
                 static_dim   = 96,
                 d_model      = 128,
                 nhead        = 4,
                 n_layers     = 2,
                 static_hidden= 64,
                 fusion_hidden= 128,
                 num_bins     = NUM_BINS,
                 dropout      = 0.2,
                 max_seq_len  = 25):
        super().__init__()
        self.d_model = d_model

        self.vital_proj = nn.Linear(vital_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_emb = nn.Embedding(max_seq_len + 1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward= d_model * 4,
            dropout        = dropout,
            batch_first    = True,
            norm_first     = True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)

        self.static_enc = nn.Sequential(
            nn.Linear(static_dim, static_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(d_model + static_hidden),
            nn.Linear(d_model + static_hidden, fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, num_bins),
        )

    def forward(self, x_seq, x_static, lengths):
        B, T, _ = x_seq.shape
        x   = self.vital_proj(x_seq)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        pos = torch.arange(T + 1, device=x.device).unsqueeze(0)
        x   = x + self.pos_emb(pos)

        mask = torch.ones(
            B, T + 1, dtype=torch.bool, device=x.device
        )
        mask[:, 0] = False
        for i, l in enumerate(lengths):
            mask[i, 1:l + 1] = False

        out     = self.transformer(x, src_key_padding_mask=mask)
        cls_out = out[:, 0, :]
        static_out = self.static_enc(x_static)
        fused  = torch.cat([cls_out, static_out], dim=1)
        logits = self.fusion(fused)
        return torch.softmax(logits, dim=1)


# ── Load calibrators ──────────────────────────────────────────
import joblib

calibrators = joblib.load(
    str(OUTPUT_DIR / 'transformer_all_calibrators.pkl')
)
print(f"Calibrators loaded : {len(calibrators)} time points")

# ── Instantiate and load weights ──────────────────────────────
model = TransformerSurvival(
    vital_dim    = len(rich_feature_names),   # 25
    static_dim   = len(feature_cols),          # 127
    d_model      = 128,
    nhead        = 4,
    n_layers     = 2,
    static_hidden= 64,
    fusion_hidden= 128,
    num_bins     = NUM_BINS,
    dropout      = 0.2,
    max_seq_len  = 25,
).to(device)

model.load_state_dict(torch.load(
    str(OUTPUT_DIR / 'transformer_all_best.pt'),
    map_location=device,
    weights_only=True,
))
model.eval()

total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Model loaded       : {total_params:,} parameters")
print(f"Device             : {device}")

# ── Calibration helper ────────────────────────────────────────
def calibrate_survival(surv, calibrators, num_bins):
    """Apply isotonic regression calibration to survival array."""
    surv_cal      = surv.copy()
    cal_t_indices = sorted(calibrators.keys())

    for t_idx in cal_t_indices:
        iso      = calibrators[t_idx]
        pred_cif = 1 - surv_cal[t_idx]
        cal_cif  = iso.predict([pred_cif])[0]
        surv_cal[t_idx] = 1 - cal_cif

    # Interpolate between calibrated time points
    x_known = np.array(cal_t_indices)
    y_known = surv_cal[cal_t_indices]
    x_all   = np.arange(num_bins)
    surv_cal = np.interp(x_all, x_known, y_known)

    # Enforce monotone decreasing
    for t in range(1, num_bins):
        if surv_cal[t] > surv_cal[t - 1]:
            surv_cal[t] = surv_cal[t - 1]

    return surv_cal

# ── Get bin index for 12h horizon ─────────────────────────────
bin_12h = int(np.clip(
    np.searchsorted(time_cuts, HERO_HORIZON_H, 'right'),
    0, NUM_BINS - 1
))
print(f"\nBin index for {HERO_HORIZON_H}h : {bin_12h} "
      f"(t={time_cuts[bin_12h]:.1f}h)")
print("\nModel ready ✓")

In [ ]:
# ============================================================
# Cell 3: Build rolling 12h-ahead labels from hourly_labels
#
# For each stay-hour (stay_id, h):
#   y_roll = 1 if sepsis onset occurs between h and h+12
#          = 0 otherwise
# ============================================================
print("Building rolling 12h-ahead labels...")

# ── Find sepsis onset hour per stay ───────────────────────────
sepsis_onset = (
    hl[hl['label'] == 1]
    .groupby('stay_id')['hour']
    .min()
    .reset_index()
    .rename(columns={'hour': 'onset_hour'})
)
onset_lookup = sepsis_onset.set_index('stay_id')['onset_hour'].to_dict()

# ── Max observed hour per stay ────────────────────────────────
max_hour_per_stay = (
    hl.groupby('stay_id')['hour']
    .max()
    .to_dict()
)

print(f"Stays with sepsis onset : {len(onset_lookup):,}")
print(f"Total stays             : {len(stay_ids_order):,}")

# ── Build label lookup: (stay_id, hour) → y_roll ─────────────
# We will use this during inference to assign labels
def get_rolling_label(stay_id, hour, horizon=HERO_HORIZON_H):
    """
    Returns 1 if sepsis onset occurs between hour and hour+horizon.
    Returns 0 if censored or onset outside window.
    Returns -1 if hour exceeds stay length (should not be evaluated).
    """
    max_h = max_hour_per_stay.get(stay_id, 0)
    if hour > max_h:
        return -1   # beyond stay — skip

    onset = onset_lookup.get(stay_id, None)
    if onset is None:
        return 0    # no sepsis — censored

    if hour <= onset < hour + horizon:
        return 1    # sepsis onset within next horizon hours
    else:
        return 0    # sepsis outside window or already occurred

# ── Quick verification ────────────────────────────────────────
print("\nLabel verification (3 sepsis stays):")
for sid in list(onset_lookup.keys())[:3]:
    onset = onset_lookup[sid]
    for h in [onset - 6, onset - 1, onset, onset + 1]:
        if h >= 0:
            lbl = get_rolling_label(sid, h)
            print(f"  stay={sid} hour={h:>4} onset={onset:>4} "
                  f"→ label={lbl}")

print("\nRolling labels ready ✓")

In [ ]:
# ============================================================
# Cell 4: Rolling inference — main loop
#
# Processes all stays in batches for efficiency.
# For each stay, slides hour by hour through X_rich_full
# using a 24h lookback window.
# Expected time: 20-40 minutes for 74,550 stays
# ============================================================
import time

print("Starting rolling inference...")
print(f"  Lookback window : {LOOKBACK_W}h")
print(f"  Prediction horizon: {HERO_HORIZON_H}h")
print(f"  Min hour        : {MIN_HOUR}")
print(f"  Max hours       : {MAX_HOURS}")
print(f"  Alert thresholds: {ALERT_THRESHOLDS}")
print()

BATCH_SIZE = 256   # process this many stay-hours at once
results    = []
t_start    = time.time()

# ── Pre-load static features as tensor ───────────────────────
print("Pre-loading static features...")
X_static_all = np.zeros(
    (len(stay_ids_order), len(feature_cols)),
    dtype=np.float32
)
for i, sid in enumerate(stay_ids_order):
    if sid in static_lookup.index:
        X_static_all[i] = static_lookup.loc[sid].values.astype(np.float32)

X_static_tensor = torch.tensor(
    X_static_all, dtype=torch.float32
)
print(f"Static features shape : {X_static_all.shape}")

# ── Build list of all (stay_idx, hour) pairs to evaluate ──────
print("Building evaluation schedule...")
eval_pairs = []   # (stay_idx, stay_id, hour)

for i, sid in enumerate(stay_ids_order):
    max_h = min(
        max_hour_per_stay.get(sid, MIN_HOUR),
        MAX_HOURS - 1
    )
    for h in range(MIN_HOUR, max_h + 1):
        eval_pairs.append((i, sid, h))

total_pairs = len(eval_pairs)
print(f"Total stay-hours to evaluate : {total_pairs:,}")
print(f"Estimated time @ {BATCH_SIZE} batch: "
      f"~{total_pairs // BATCH_SIZE // 60 + 1} min\n")

# ── Batch inference ───────────────────────────────────────────
model.eval()

for batch_start in range(0, total_pairs, BATCH_SIZE):
    batch = eval_pairs[batch_start:batch_start + BATCH_SIZE]

    # Build batch tensors
    x_seqs   = []
    x_statics= []
    lengths  = []
    meta     = []   # (stay_id, hour, split)

    for stay_idx, stay_id, hour in batch:
        # Lookback window: last LOOKBACK_W hours up to hour
        h_start = max(0, hour - LOOKBACK_W)
        seq     = X_rich_full[stay_idx, h_start:hour, :]
        seq_len = len(seq)

        if seq_len == 0:
            continue

        x_seqs.append(torch.tensor(seq, dtype=torch.float32))
        x_statics.append(X_static_tensor[stay_idx])
        lengths.append(seq_len)
        meta.append((stay_id, hour, stay_split.get(stay_id, 'unknown')))

    if not x_seqs:
        continue

    # Pad sequences to same length
    max_len  = max(lengths)
    x_padded = torch.zeros(
        len(x_seqs), max_len, len(rich_feature_names)
    )
    for i, seq in enumerate(x_seqs):
        x_padded[i, :len(seq), :] = seq

    x_padded  = x_padded.to(device)
    x_static_b= torch.stack(x_statics).to(device)
    lengths_t = torch.tensor(lengths, dtype=torch.long).to(device)

    # Inference
    with torch.no_grad():
        pmf = model(x_padded, x_static_b, lengths_t).cpu().numpy()

    # Convert PMF → survival → calibrated risk
    for i, (sid, hour, split) in enumerate(meta):
        cif     = np.cumsum(pmf[i])
        surv    = 1 - cif
        surv_cal= calibrate_survival(surv, calibrators, NUM_BINS)
        risk_12h= float(1 - surv_cal[bin_12h])
        risk_12h= np.clip(risk_12h, 0.0, 1.0)
        tier    = get_alert_tier(risk_12h)
        label   = get_rolling_label(sid, hour)

        if label == -1:
            continue   # beyond stay length — skip

        results.append({
            'stay_id'       : sid,
            'hour'          : hour,
            'risk_score'    : round(risk_12h, 4),
            'alert_tier'    : tier,
            'true_label_12h': label,
            'split'         : split,
        })

    # Progress
    if (batch_start // BATCH_SIZE) % 500 == 0:
        elapsed = time.time() - t_start
        pct     = batch_start / total_pairs
        eta     = (elapsed / max(pct, 1e-6)) * (1 - pct) / 60
        print(f"  {batch_start:>8,} / {total_pairs:,} "
              f"({pct:.1%}) | "
              f"elapsed={elapsed/60:.1f}min | "
              f"ETA={eta:.1f}min",
              end='\r')

print(f"\n\nInference complete.")
print(f"Total time : {(time.time()-t_start)/60:.1f} minutes")
print(f"Results    : {len(results):,} stay-hour predictions")

# ── Save ──────────────────────────────────────────────────────
rolling_df = pd.DataFrame(results)
rolling_df.to_csv(
    str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False
)
print(f"\nSaved → rolling_predictions.csv ✓")
print(f"Shape  : {rolling_df.shape}")
print(f"\nPreview:")
print(rolling_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Cell 5: Validate rolling predictions
# ============================================================
print("Validating rolling predictions...")
print(f"\nShape : {rolling_df.shape}")

# ── Distribution of alert tiers ───────────────────────────────
print("\nAlert tier distribution:")
tier_counts = rolling_df['alert_tier'].value_counts()
for tier, count in tier_counts.items():
    pct = count / len(rolling_df)
    print(f"  {tier:<12} : {count:>8,}  ({pct:.2%})")

# ── Risk score distribution ───────────────────────────────────
print(f"\nRisk score stats:")
print(f"  Mean   : {rolling_df['risk_score'].mean():.4f}")
print(f"  Median : {rolling_df['risk_score'].median():.4f}")
print(f"  Std    : {rolling_df['risk_score'].std():.4f}")
print(f"  Min    : {rolling_df['risk_score'].min():.4f}")
print(f"  Max    : {rolling_df['risk_score'].max():.4f}")

# ── Label distribution ────────────────────────────────────────
print(f"\nLabel distribution:")
lbl_counts = rolling_df['true_label_12h'].value_counts()
print(f"  Positive (sepsis within 12h) : "
      f"{lbl_counts.get(1,0):>8,} "
      f"({lbl_counts.get(1,0)/len(rolling_df):.3%})")
print(f"  Negative                     : "
      f"{lbl_counts.get(0,0):>8,} "
      f"({lbl_counts.get(0,0)/len(rolling_df):.3%})")

# ── AUROC on test set ─────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

test_df = rolling_df[rolling_df['split'] == 'test']
print(f"\nTest set rolling evaluation:")
print(f"  Stay-hours : {len(test_df):,}")

auroc = roc_auc_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
auprc = average_precision_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")

# ── Per-split breakdown ───────────────────────────────────────
print(f"\nPer-split breakdown:")
for split in ['train', 'val', 'test']:
    sub = rolling_df[rolling_df['split'] == split]
    if len(sub) == 0:
        continue
    pos_rate = sub['true_label_12h'].mean()
    print(f"  {split:<6} : {len(sub):>8,} stay-hours | "
          f"positive rate={pos_rate:.3%}")

# ── Sample patient timeline ───────────────────────────────────
print(f"\nSample patient timeline (first sepsis patient):")
sepsis_stays = rolling_df[
    rolling_df['true_label_12h'] == 1
]['stay_id'].unique()

if len(sepsis_stays) > 0:
    sample_sid = sepsis_stays[0]
    sample     = rolling_df[
        rolling_df['stay_id'] == sample_sid
    ].sort_values('hour')
    onset      = onset_lookup.get(sample_sid, '?')
    print(f"  stay_id={sample_sid}  onset={onset}h")
    print(f"  {'Hour':>5} {'Risk':>8} {'Tier':<12} {'Label':>6}")
    print(f"  {'-'*35}")
    for _, row in sample.iterrows():
        marker = ' ← ONSET' if row['hour'] == onset else ''
        print(f"  {row['hour']:>5.0f} "
              f"{row['risk_score']:>8.4f} "
              f"{row['alert_tier']:<12} "
              f"{row['true_label_12h']:>6}{marker}")

print("\n03d complete ✓")
print("Ready for updated 04 — rolling evaluation")